# Analyze asset topology

This notebook examines Powsybl topologies at voltage-level, bus-group, and element level. It uses `pypowsybl_jupyter` for zoomable single-line diagrams and keeps canonical master data strictly separate from the materialized runtime topology.

For a processed ToOp data folder, it uses the available topology artifacts. For an XIIDM file, it derives the topology in memory only; source data is not modified.

## 1. Import project dependencies

In [ ]:
from __future__ import annotations

import json
from dataclasses import dataclass, replace
from enum import StrEnum
from pathlib import Path
from typing import Any
from xml.etree import ElementTree

import ipywidgets as widgets
import numpy as np
import pandas as pd
import pypowsybl
from IPython.display import HTML, display
from toop_engine_dc_solver.preprocess.preprocess_switching import StationProblems, prepare_for_separation_set
from toop_engine_dc_solver.preprocess.simplify_topology import _project_station_to_local_assets
from toop_engine_grid_helpers.powsybl.example_grids import create_default_network_masks
from toop_engine_grid_helpers.powsybl.powsybl_asset_topo import (
    get_bus_breaker_master_asset_topology,
    materialize_runtime_bus_groups_from_network_state,
)
from toop_engine_grid_helpers.powsybl.powsybl_station_to_graph import get_node_breaker_master_asset_topology
from toop_engine_grid_helpers.powsybl.single_line_diagram.get_single_line_diagram_custom import (
    get_single_line_diagram_custom,
)
from toop_engine_interfaces.asset_topology.asset_topology import MasterAssetTopology
from toop_engine_interfaces.asset_topology.runtime_topology import RuntimeAssetConnection, RuntimeAssetTopology, RuntimeBusGroup
from toop_engine_interfaces.asset_topology.simplified_runtime_topology import SimplifiedBusGroup
from toop_engine_interfaces.folder_structure import PREPROCESSING_PATHS
from toop_engine_interfaces.messages.preprocess.preprocess_commands import AreaSettings, CgmesImporterParameters

## 2. Define configuration and paths

In [ ]:
class SourceMode(StrEnum):
    """Supported topology input modes."""

    AUTO = "auto"
    PROCESSED_FOLDER = "processed data folder"
    XIIDM_FILE = "XIIDM file"


DEFAULT_SOURCE_PATH = Path("../data/grid_node_breaker/grid.xiidm")
SLD_STYLE = "bright_mode"
FOCUS_COLOR = "#0050fc"
MUTED_OPACITY = "0.14"


@dataclass(frozen=True)
class TopologySource:
    """Resolved input source for topology analysis."""

    grid_path: Path
    source_root: Path
    mode: SourceMode
    master_data_path: Path | None = None
    runtime_topology_path: Path | None = None

## 3. Create data models and interfaces

In [ ]:
@dataclass
class TopologyContext:
    """Powsybl network and its separate canonical and runtime topology views."""

    source: TopologySource
    network: Any
    master_data: MasterAssetTopology
    runtime_topology: RuntimeAssetTopology

    @property
    def stations_by_id(self) -> dict[str, RuntimeBusGroup]:
        """Return materialized station views by stable bus-group id."""
        return {station.bus_group_id: station for station in self.runtime_topology.stations}


@dataclass(frozen=True)
class ElementSelection:
    """One station-local asset or coupler selected for inspection."""

    kind: str
    grid_model_id: str
    label: str
    connection_index: int | None = None


@dataclass(frozen=True)
class SimplificationEvent:
    """One object-level difference introduced by in-memory simplification."""

    category: str
    object_type: str
    grid_model_id: str
    reason: str
    retained_grid_model_id: str | None = None


@dataclass(frozen=True)
class StationSimplificationComparison:
    """Separate original and simplified runtime views for one bus group."""

    original_station: RuntimeBusGroup
    simplified_station: SimplifiedBusGroup
    problems: StationProblems
    events: tuple[SimplificationEvent, ...]


def resolve_source(source_path: str | Path, mode: SourceMode = SourceMode.AUTO) -> TopologySource:
    """Resolve a processed data folder or a raw XIIDM input file."""
    path = Path(source_path).expanduser().resolve()
    if not path.exists():
        raise FileNotFoundError(f"Input path does not exist: {path}")

    if path.is_file():
        if mode == SourceMode.PROCESSED_FOLDER:
            raise ValueError("Processed data folder mode requires a directory.")
        return TopologySource(grid_path=path, source_root=path.parent, mode=SourceMode.XIIDM_FILE)

    grid_path = path / PREPROCESSING_PATHS["grid_file_path_powsybl"]
    master_path = path / PREPROCESSING_PATHS["asset_topology_master_data_file_path"]
    runtime_path = path / PREPROCESSING_PATHS["asset_topology_runtime_file_path"]
    if not grid_path.exists():
        raise FileNotFoundError(f"Processed data folder does not contain {grid_path.name}: {path}")
    if mode == SourceMode.XIIDM_FILE:
        raise ValueError("XIIDM file mode requires a file path.")
    return TopologySource(
        grid_path=grid_path,
        source_root=path,
        mode=SourceMode.PROCESSED_FOLDER,
        master_data_path=master_path,
        runtime_topology_path=runtime_path,
    )


def load_json_model(path: Path, model_type: type[Any]) -> Any:
    """Load one Pydantic topology artifact without coercing it into another model."""
    return model_type.model_validate(json.loads(path.read_text(encoding="utf-8")))

## 4. Create the implementation scaffold

In [ ]:
def build_topology_from_xiidm(source: TopologySource, network: Any) -> tuple[MasterAssetTopology, RuntimeAssetTopology]:
    """Build canonical and materialized topology views in memory for a raw XIIDM file."""
    topology_kinds = set(network.get_voltage_levels(attributes=["topology_kind"])["topology_kind"].dropna())
    if "NODE_BREAKER" in topology_kinds:
        network_masks = create_default_network_masks(network)
        network_masks = replace(
            network_masks,
            relevant_subs=np.ones(len(network.get_buses()), dtype=bool),
            busbar_for_nminus1=np.ones(len(network.get_busbar_sections()), dtype=bool),
        )
        master_data = get_node_breaker_master_asset_topology(
            network=network,
            network_masks=network_masks,
            importer_parameters=CgmesImporterParameters(
                area_settings=AreaSettings(control_area=[""], view_area=[""], nminus1_area=[""], cutoff_voltage=1),
                data_folder=source.source_root,
                grid_model_file=source.grid_path,
            ),
        )
    else:
        master_data = get_bus_breaker_master_asset_topology(
            network=network,
            relevant_stations=list(network.get_buses().index),
            topology_id=source.grid_path.name,
        )

    runtime_stations = materialize_runtime_bus_groups_from_network_state(network=network, master_data=master_data)
    return master_data, RuntimeAssetTopology(stations=runtime_stations, circuit_groups=master_data.circuit_groups)


def load_topology_context(source_path: str | Path, mode: SourceMode = SourceMode.AUTO) -> TopologyContext:
    """Load the Powsybl network plus separate canonical and runtime topology representations."""
    source = resolve_source(source_path, mode)
    network = pypowsybl.network.load(str(source.grid_path))
    if source.mode == SourceMode.PROCESSED_FOLDER:
        if source.master_data_path is None or not source.master_data_path.exists():
            raise FileNotFoundError("Missing canonical master topology artifact in initial_topology.")
        master_data = load_json_model(source.master_data_path, MasterAssetTopology)
        if source.runtime_topology_path is not None and source.runtime_topology_path.exists():
            runtime_topology = load_json_model(source.runtime_topology_path, RuntimeAssetTopology)
        else:
            runtime_stations = materialize_runtime_bus_groups_from_network_state(network=network, master_data=master_data)
            runtime_topology = RuntimeAssetTopology(stations=runtime_stations, circuit_groups=master_data.circuit_groups)
    else:
        master_data, runtime_topology = build_topology_from_xiidm(source, network)
    return TopologyContext(source=source, network=network, master_data=master_data, runtime_topology=runtime_topology)

In [ ]:
def get_voltage_levels(context: TopologyContext) -> pd.DataFrame:
    """Return voltage levels with stable ids and display names."""
    voltage_levels = context.network.get_voltage_levels(attributes=["name", "topology_kind", "nominal_v"])
    voltage_levels = voltage_levels.copy()
    voltage_levels["display_name"] = voltage_levels["name"].fillna(voltage_levels.index.to_series())
    return voltage_levels


def get_stations_for_voltage_level(context: TopologyContext, voltage_level_id: str) -> list[RuntimeBusGroup]:
    """Return materialized runtime bus groups for one voltage level."""
    return [station for station in context.runtime_topology.stations if station.voltage_level_id == voltage_level_id]


def get_element_selections(station: RuntimeBusGroup) -> list[ElementSelection]:
    """Return station-local branches, injections, and couplers as inspectable selections."""
    selections: list[ElementSelection] = []
    for index, connection in enumerate(station.branch_connections):
        asset = connection.asset
        selections.append(ElementSelection("branch", asset.grid_model_id, f"Branch: {asset.name or asset.grid_model_id}", index))
    for index, connection in enumerate(station.injection_connections):
        asset = connection.asset
        selections.append(ElementSelection("injection", asset.grid_model_id, f"Injection: {asset.name or asset.grid_model_id}", index))
    for coupler in station.couplers:
        selections.append(ElementSelection("coupler", coupler.grid_model_id, f"Coupler: {coupler.name or coupler.grid_model_id}"))
    return selections


def get_asset_bay_switch_ids(connection: RuntimeAssetConnection) -> set[str]:
    """Return all switch ids that physically implement one asset connection."""
    if connection.asset_bay is None:
        return set()
    asset_bay = connection.asset_bay
    return {
        asset_bay.dv_switch_grid_model_id,
        *asset_bay.busbar_disconnector_grid_model_id.values(),
        *([asset_bay.asset_disconnector_grid_model_id] if asset_bay.asset_disconnector_grid_model_id else []),
    }


def get_coupler_switch_ids(coupler: Any) -> set[str]:
    """Return every physical switch id recorded for one runtime coupler."""
    switch_ids: set[str] = set()
    if coupler.asset_bay is not None:
        switch_ids.update(get_asset_bay_switch_ids(coupler))
    if coupler.coupler_bay is not None:
        coupler_bay = coupler.coupler_bay
        switch_ids.update(coupler_bay.coupler_breaker_ids)
        switch_ids.update(coupler_bay.coupler_disconnector_ids)
        switch_ids.update(coupler_bay.from_busbar_disconnector_ids.values())
        switch_ids.update(coupler_bay.to_busbar_disconnector_ids.values())
    return switch_ids


def get_station_focus_ids(station: RuntimeBusGroup) -> set[str]:
    """Return every station element and physical switch shown in a bus-group SLD view."""
    focus_ids = {
        *(busbar.grid_model_id for busbar in station.busbars),
        *(coupler.grid_model_id for coupler in station.couplers),
        *(connection.asset.grid_model_id for connection in station.branch_connections),
        *(connection.asset.grid_model_id for connection in station.injection_connections),
    }
    for connection in [*station.branch_connections, *station.injection_connections]:
        focus_ids.update(get_asset_bay_switch_ids(connection))
    for coupler in station.couplers:
        focus_ids.update(get_coupler_switch_ids(coupler))
    return focus_ids


def get_voltage_level_focus_ids(context: TopologyContext, voltage_level_id: str) -> set[str]:
    """Return all topology element and bay-switch ids currently shown for one voltage level."""
    focus_ids: set[str] = set()
    for station in get_stations_for_voltage_level(context, voltage_level_id):
        focus_ids.update(get_station_focus_ids(station))
    return focus_ids


def get_master_element_payload(
    context: TopologyContext,
    station_id: str,
    selection: ElementSelection,
) -> dict[str, Any]:
    """Return canonical topology-owned data for a selected runtime element without mixing runtime fields."""
    master_station = next(station for station in context.master_data.stations if station.bus_group_id == station_id)
    if selection.kind == "coupler":
        coupler = next(coupler for coupler in master_station.couplers if coupler.grid_model_id == selection.grid_model_id)
        return {"coupler": coupler.model_dump(mode="json")}

    connections = master_station.branch_connections if selection.kind == "branch" else master_station.injection_connections
    connection = connections[selection.connection_index]
    assets = context.master_data.branch_assets if selection.kind == "branch" else context.master_data.injection_assets
    asset = next(asset for asset in assets if asset.grid_model_id == selection.grid_model_id)
    asset_bays = {asset_bay.asset_bay_id: asset_bay for asset_bay in context.master_data.asset_bays}
    return {
        "asset": asset.model_dump(mode="json"),
        "station connection": connection.model_dump(mode="json"),
        "asset bay": asset_bays[connection.asset_bay_id].model_dump(mode="json") if connection.asset_bay_id else None,
    }


def as_json_view(payload: Any) -> widgets.Textarea:
    """Render complete structured data in a readable, scrollable JSON pane."""
    if hasattr(payload, "model_dump"):
        payload = payload.model_dump(mode="json")
    return widgets.Textarea(
        value=json.dumps(payload, indent=2, ensure_ascii=True),
        disabled=True,
        layout=widgets.Layout(width="100%", height="440px"),
    )

In [ ]:
def as_key_value_frame(payload: Any) -> widgets.Textarea:
    """Preserve the existing element-pane call sites with a full JSON view."""
    return as_json_view(payload)

In [ ]:
def get_station_object_ids(station: RuntimeBusGroup) -> dict[str, set[str]]:
    """Return station objects grouped by stable grid-model identity."""
    return {
        "busbar": {busbar.grid_model_id for busbar in station.busbars},
        "coupler": {coupler.grid_model_id for coupler in station.couplers},
        "branch": {connection.asset.grid_model_id for connection in station.branch_connections},
        "injection": {connection.asset.grid_model_id for connection in station.injection_connections},
    }


def get_simplification_events(
    original_station: RuntimeBusGroup,
    simplified_station: SimplifiedBusGroup,
    problems: StationProblems,
) -> tuple[SimplificationEvent, ...]:
    """Describe simplification differences supported by stable ids or production diagnostics."""
    original_ids = get_station_object_ids(original_station)
    simplified_ids = get_station_object_ids(simplified_station)
    duplicate_coupler_ids = {coupler.grid_model_id for coupler in problems.duplicate_couplers or []}
    disconnected_busbar_ids = {busbar.grid_model_id for busbar in problems.disconnected_busbars or []}
    original_couplers = {coupler.grid_model_id: coupler for coupler in original_station.couplers}
    original_busbars_by_int_id = {busbar.int_id: busbar.grid_model_id for busbar in original_station.busbars}
    retained_busbar_ids = simplified_ids["busbar"]
    events: list[SimplificationEvent] = []

    for object_type in ("branch", "injection"):
        for grid_model_id in sorted(original_ids[object_type] - simplified_ids[object_type]):
            events.append(SimplificationEvent("removed", object_type, grid_model_id, "removed during simplification"))

    for grid_model_id in sorted(original_ids["busbar"] - simplified_ids["busbar"]):
        reason = "disconnected busbar" if grid_model_id in disconnected_busbar_ids else "removed during simplification"
        events.append(SimplificationEvent("removed", "busbar", grid_model_id, reason))

    for grid_model_id in sorted(original_ids["coupler"] - simplified_ids["coupler"]):
        coupler = original_couplers[grid_model_id]
        adjacent_busbars = {
            original_busbars_by_int_id[coupler.busbar_from_id],
            original_busbars_by_int_id[coupler.busbar_to_id],
        }
        removed_busbars = adjacent_busbars - retained_busbar_ids
        retained_adjacent_busbars = adjacent_busbars & retained_busbar_ids
        if coupler.coupler_type == "DISCONNECTOR" and len(removed_busbars) == 1 and len(retained_adjacent_busbars) == 1:
            events.append(
                SimplificationEvent(
                    "fused",
                    "disconnector",
                    grid_model_id,
                    f"absorbed busbar {next(iter(removed_busbars))}",
                    next(iter(retained_adjacent_busbars)),
                )
            )
        else:
            reason = "duplicate coupler" if grid_model_id in duplicate_coupler_ids else "removed during simplification"
            events.append(SimplificationEvent("removed", "coupler", grid_model_id, reason))

    for asset, removed_busbar, retained_busbar in problems.multi_connected_assets or []:
        events.append(
            SimplificationEvent(
                "reconnected",
                "asset",
                asset.grid_model_id,
                f"removed connection to {removed_busbar.grid_model_id} without an intervening coupler",
                retained_busbar.grid_model_id,
            )
        )
    return tuple(events)


def simplify_station_for_analysis(station: RuntimeBusGroup) -> StationSimplificationComparison:
    """Reproduce the default preprocessing simplification for one runtime bus group in memory."""
    branch_ids = [connection.asset.grid_model_id for connection in station.branch_connections]
    injection_ids = [connection.asset.grid_model_id for connection in station.injection_connections]
    original_station = _project_station_to_local_assets(
        station=station,
        branch_ids=branch_ids,
        injection_ids=injection_ids,
    )
    simplified_station, problems = prepare_for_separation_set(
        station=original_station,
        branch_ids=branch_ids,
        injection_ids=injection_ids,
        close_couplers=False,
    )
    return StationSimplificationComparison(
        original_station=original_station,
        simplified_station=simplified_station,
        problems=problems,
        events=get_simplification_events(original_station, simplified_station, problems),
    )

In [ ]:
def summarize_simplification(context: TopologyContext) -> pd.DataFrame:
    """Return one non-mutating simplification summary for the first runtime bus group."""
    analysis_station = context.runtime_topology.stations[0]
    runtime_topology_before_analysis = context.runtime_topology.model_dump(mode="json")
    analysis_comparison = simplify_station_for_analysis(analysis_station)

    assert isinstance(analysis_comparison.original_station, RuntimeBusGroup)
    assert isinstance(analysis_comparison.simplified_station, SimplifiedBusGroup)
    assert context.runtime_topology.model_dump(mode="json") == runtime_topology_before_analysis

    original_ids = get_station_object_ids(analysis_comparison.original_station)
    simplified_ids = get_station_object_ids(analysis_comparison.simplified_station)
    for event in analysis_comparison.events:
        object_type = "coupler" if event.object_type == "disconnector" else event.object_type
        assert event.grid_model_id in original_ids[object_type]
        if event.category == "removed":
            assert event.grid_model_id not in simplified_ids[object_type]
        if event.category == "fused":
            assert event.retained_grid_model_id in simplified_ids["busbar"]

    return pd.DataFrame(
        [
            {
                "bus group": analysis_station.bus_group_id,
                "original busbars": len(analysis_comparison.original_station.busbars),
                "simplified busbars": len(analysis_comparison.simplified_station.busbars),
                "original couplers": len(analysis_comparison.original_station.couplers),
                "simplified couplers": len(analysis_comparison.simplified_station.couplers),
                "events": len(analysis_comparison.events),
            }
        ]
    )

In [ ]:
def run_explorer_simplification_analysis(app: Any) -> pd.DataFrame:
    """Run and summarize in-memory simplification for the current explorer voltage level."""
    assert app.context is not None
    runtime_topology_before_analysis = app.context.runtime_topology.model_dump(mode="json")
    app._run_simplification_analysis(app.run_simplification_button)

    assert app.simplification_cache
    assert app.context.runtime_topology.model_dump(mode="json") == runtime_topology_before_analysis
    assert len(app.tabs.children) == 4
    assert app.tabs.get_title(3) == "Simplification"

    comparison_rows = [
        {
            "bus group": bus_group_id,
            "events": len(comparison.events),
            "busbars": f"{len(comparison.original_station.busbars)} -> {len(comparison.simplified_station.busbars)}",
            "couplers": f"{len(comparison.original_station.couplers)} -> {len(comparison.simplified_station.couplers)}",
        }
        for bus_group_id, comparison in app.simplification_cache.items()
    ]
    comparison_frame = pd.DataFrame(comparison_rows).sort_values(["events", "bus group"], ascending=[False, True])
    changed_bus_group_id = comparison_frame.iloc[0]["bus group"]
    app.simplification_bus_group.value = changed_bus_group_id
    sld_widget = app.clickable_sld_widgets["simplification"] if app.clickable_sld.value else app.sld_widgets["simplification"]
    assert sld_widget is not None
    return comparison_frame

In [ ]:
def get_sld_content(context: TopologyContext, voltage_level_id: str, focus_ids: set[str] | None = None) -> str:
    """Generate an untouched Powsybl SLD with its native focused-element styling."""
    svg = get_single_line_diagram_custom(
        net=context.network,
        container_id=voltage_level_id,
        custom_style=SLD_STYLE,
        highlight_grid_model_ids=sorted(focus_ids) if focus_ids else None,
        highlight_color=FOCUS_COLOR,
    )
    return svg._content


def is_grid_model_id_in_sld(svg_content: str, grid_model_id: str) -> bool:
    """Return whether Powsybl emitted a grid-model id using its SVG attribute encoding."""
    svg_id = "id" + "".join(character if character.isalnum() else f"_{ord(character)}_" for character in grid_model_id)
    return svg_id in svg_content


def update_or_create_sld(
    current_widget: Any | None,
    svg_content: str,
    on_hover: Any,
) -> widgets.Image:
    """Render an untouched Powsybl SLD at a consistent viewport height."""
    del on_hover
    svg_bytes = svg_content.encode("utf-8")
    if isinstance(current_widget, widgets.Image):
        current_widget.value = svg_bytes
        return current_widget
    return widgets.Image(
        value=svg_bytes,
        format="svg+xml",
        layout=widgets.Layout(width="100%", height="560px", object_fit="contain"),
    )

## 5. Load and validate example data

Enter a processed ToOp data folder or an XIIDM file. The loading step validates that the canonical and runtime topologies are available separately.

In [ ]:
def validate_context(context: TopologyContext) -> pd.DataFrame:
    """Validate the minimum data required by all notebook views."""
    if not context.runtime_topology.stations:
        raise ValueError("Runtime topology contains no materialized bus groups.")
    master_station_ids = {station.bus_group_id for station in context.master_data.stations}
    runtime_station_ids = {station.bus_group_id for station in context.runtime_topology.stations}
    missing_master_stations = runtime_station_ids - master_station_ids
    if missing_master_stations:
        raise ValueError(f"Runtime station ids missing from master data: {sorted(missing_master_stations)}")

    return pd.DataFrame(
        [
            ("source", str(context.source.grid_path)),
            ("source mode", context.source.mode.value),
            ("master bus groups", len(context.master_data.stations)),
            ("runtime bus groups", len(context.runtime_topology.stations)),
            ("branch assets", len(context.master_data.branch_assets)),
            ("injection assets", len(context.master_data.injection_assets)),
            ("asset bays", len(context.master_data.asset_bays)),
        ],
        columns=["check", "value"],
    )

In [ ]:
# The default points to the versioned Node-Breaker XIIDM fixture. Change the path above before running this cell for another grid.
example_context = load_topology_context(DEFAULT_SOURCE_PATH, SourceMode.XIIDM_FILE)
validate_context(example_context)

## 6. Run the initial processing

In [ ]:
class AssetTopologyExplorer:
    """Interactive Powsybl workspace for runtime asset-topology inspection."""

    def __init__(self) -> None:
        self.context: TopologyContext | None = None
        self.sld_widgets: dict[str, widgets.Image | None] = {
            "voltage_level": None,
            "bus_group": None,
            "element": None,
            "simplification": None,
        }
        self.clickable_sld_widgets: dict[str, Any | None] = {key: None for key in self.sld_widgets}
        self.simplification_cache: dict[str, StationSimplificationComparison] = {}
        self.source_path = widgets.Text(value=str(DEFAULT_SOURCE_PATH), description="Input path", layout=widgets.Layout(width="720px"))
        self.source_mode = widgets.Dropdown(
            options=[(mode.value, mode) for mode in SourceMode], value=SourceMode.AUTO, description="Source mode"
        )
        self.load_button = widgets.Button(description="Load topology", icon="folder-open", button_style="primary")
        self.clickable_sld = widgets.ToggleButton(
            value=True,
            description="Clickable SLD",
            icon="mouse-pointer",
            tooltip="Right-click a switch or feeder to open its element details. Disable this for the static SVG fallback.",
        )
        self.voltage_level = widgets.Dropdown(description="Voltage level", layout=widgets.Layout(width="560px"))
        self.bus_group = widgets.Dropdown(description="Bus group", layout=widgets.Layout(width="560px"))
        self.element = widgets.Dropdown(description="Element", layout=widgets.Layout(width="560px"))
        self.simplification_bus_group = widgets.Dropdown(description="Bus group", layout=widgets.Layout(width="560px"))
        self.run_simplification_button = widgets.Button(
            description="Run simplification analysis", icon="play", button_style="primary"
        )
        self.status = widgets.Output()
        self.summary_output = widgets.Output()
        self.voltage_level_sld = widgets.VBox(layout=widgets.Layout(width="100%"))
        self.bus_group_sld = widgets.VBox(layout=widgets.Layout(width="100%"))
        self.bus_group_output = widgets.Output()
        self.element_sld = widgets.VBox(layout=widgets.Layout(width="100%"))
        data_pane_layout = widgets.Layout(width="50%", min_width="0")
        self.master_output = widgets.Output(layout=data_pane_layout)
        self.runtime_output = widgets.Output(layout=data_pane_layout)
        self.data_panes = widgets.HBox(
            [self.master_output, self.runtime_output],
            layout=widgets.Layout(width="100%", align_items="stretch"),
        )
        self.simplification_summary_output = widgets.Output()
        self.simplification_sld = widgets.VBox(layout=widgets.Layout(width="100%"))
        self.simplification_events_output = widgets.Output()
        self.simplification_original_output = widgets.Output(layout=data_pane_layout)
        self.simplification_result_output = widgets.Output(layout=data_pane_layout)
        self.simplification_panes = widgets.HBox(
            [self.simplification_original_output, self.simplification_result_output],
            layout=widgets.Layout(width="100%", align_items="stretch"),
        )
        self.tabs = widgets.Tab()
        self.load_button.on_click(self._load_context)
        self.run_simplification_button.on_click(self._run_simplification_analysis)
        self.clickable_sld.observe(self._on_clickable_sld_change, names="value")
        self.voltage_level.observe(self._on_voltage_level_change, names="value")
        self.bus_group.observe(self._on_bus_group_change, names="value")
        self.element.observe(self._on_element_change, names="value")
        self.simplification_bus_group.observe(self._on_simplification_bus_group_change, names="value")
        self.tabs.observe(self._on_tab_change, names="selected_index")

    def show(self) -> None:
        """Display the complete tabbed analysis UI."""
        voltage_tab = widgets.VBox([self.voltage_level, self.voltage_level_sld, self.summary_output])
        bus_group_tab = widgets.VBox([self.bus_group, self.bus_group_sld, self.bus_group_output])
        element_tab = widgets.VBox([self.element, self.element_sld, self.data_panes], layout=widgets.Layout(width="100%"))
        simplification_tab = widgets.VBox(
            [
                widgets.HBox([self.simplification_bus_group, self.run_simplification_button]),
                self.simplification_summary_output,
                self.simplification_sld,
                self.simplification_events_output,
                self.simplification_panes,
            ],
            layout=widgets.Layout(width="100%"),
        )
        self.tabs.children = [voltage_tab, bus_group_tab, element_tab, simplification_tab]
        self.tabs.set_title(0, "Voltage level")
        self.tabs.set_title(1, "Bus groups")
        self.tabs.set_title(2, "Elements and bays")
        self.tabs.set_title(3, "Simplification")
        display(
            widgets.VBox(
                [widgets.HBox([self.source_path, self.source_mode, self.load_button, self.clickable_sld]), self.status, self.tabs]
            )
        )

    def _load_context(self, _button: widgets.Button) -> None:
        with self.status:
            self.status.clear_output()
            try:
                self.context = load_topology_context(self.source_path.value, self.source_mode.value)
                self.simplification_cache.clear()
                self.clickable_sld_widgets = {key: None for key in self.sld_widgets}
                display(validate_context(self.context))
            except Exception as error:
                self.context = None
                display(HTML(f"<b>Could not load topology:</b> {error}"))
                return
        voltage_levels = get_voltage_levels(self.context)
        self.voltage_level.options = [(row.display_name, voltage_level_id) for voltage_level_id, row in voltage_levels.iterrows()]
        if self.voltage_level.options:
            self.voltage_level.value = self.voltage_level.options[0][1]
            self._on_voltage_level_change({})

    def _on_tab_change(self, _change: dict[str, Any]) -> None:
        """Render the active tab even when its selector value did not change."""
        if self.context is None:
            return
        if self.tabs.selected_index == 0:
            self._render_voltage_level()
        elif self.tabs.selected_index == 1:
            station = self._selected_station()
            if station is not None:
                self._render_bus_group(station)
        elif self.tabs.selected_index == 2:
            station = self._selected_station()
            selection = self.element.value
            if station is not None and selection is not None:
                self._render_element(station, selection)
        elif self.tabs.selected_index == 3:
            self._render_simplification_summary()
            self._render_simplification_detail()

    def _on_clickable_sld_change(self, _change: dict[str, Any]) -> None:
        """Refresh the active tab when changing between interactive and static SLD rendering."""
        self._on_tab_change({})

    def _on_voltage_level_change(self, _change: dict[str, Any]) -> None:
        if self.context is None or self.voltage_level.value is None:
            return
        stations = get_stations_for_voltage_level(self.context, self.voltage_level.value)
        station_options = [(station.bus_group_id, station.bus_group_id) for station in stations]
        self.bus_group.options = station_options
        self.simplification_bus_group.options = station_options
        self._render_voltage_level()
        if self.bus_group.options:
            self.bus_group.value = self.bus_group.options[0][1]
            self._on_bus_group_change({})
            self.simplification_bus_group.value = self.simplification_bus_group.options[0][1]
        else:
            self.bus_group_sld.children = ()
            self.element.options = []
            self.element_sld.children = ()
            self.master_output.clear_output()
            self.runtime_output.clear_output()
            self._clear_simplification_view()
            with self.bus_group_output:
                self.bus_group_output.clear_output(wait=True)
                if self.context.source.mode == SourceMode.PROCESSED_FOLDER:
                    display(
                        HTML(
                            "<b>No bus group is stored for this voltage level.</b><br>"
                            "It is outside the preprocessing relevance mask and therefore absent from the canonical "
                            "master topology. The original XIIDM model can still contain its buses and assets."
                        )
                    )
                else:
                    display(HTML("<b>No bus group was derived for this voltage level.</b>"))

    def _on_bus_group_change(self, _change: dict[str, Any]) -> None:
        station = self._selected_station()
        if station is None:
            return
        selections = get_element_selections(station)
        self.element.options = [(selection.label, selection) for selection in selections]
        self._render_bus_group(station)
        if self.element.options:
            self.element.value = self.element.options[0][1]
            self._on_element_change({})
        else:
            self.element_sld.children = ()
            self.master_output.clear_output()
            self.runtime_output.clear_output()

    def _on_element_change(self, _change: dict[str, Any]) -> None:
        station = self._selected_station()
        selection = self.element.value
        if station is not None and selection is not None:
            self._render_element(station, selection)

    def _on_simplification_bus_group_change(self, _change: dict[str, Any]) -> None:
        self._render_simplification_detail()

    def _selected_station(self) -> RuntimeBusGroup | None:
        if self.context is None or self.bus_group.value is None:
            return None
        return self.context.stations_by_id.get(self.bus_group.value)

    def _selected_simplification_station(self) -> RuntimeBusGroup | None:
        if self.context is None or self.simplification_bus_group.value is None:
            return None
        return self.context.stations_by_id.get(self.simplification_bus_group.value)

    def _hover_info(self, equipment_id: str, equipment_type: str) -> str:
        station = self._selected_station()
        if station is not None:
            for selection in get_element_selections(station):
                if selection.grid_model_id == equipment_id:
                    return f"<b>{selection.kind}</b><br>{selection.label}"
        return f"<b>{equipment_type}</b><br>{equipment_id}"

    def _get_clickable_sld(self, focus_ids: set[str]) -> Any:
        """Build a metadata-preserving SVG for the native Powsybl interactive widget."""
        assert self.context is not None
        return get_single_line_diagram_custom(
            net=self.context.network,
            container_id=self.voltage_level.value,
            custom_style=SLD_STYLE,
            highlight_grid_model_ids=sorted(focus_ids),
            highlight_color=FOCUS_COLOR,
        )

    def _find_clicked_element(self, grid_model_id: str) -> tuple[RuntimeBusGroup, ElementSelection] | None:
        """Resolve a clicked SLD asset, coupler, or physical bay switch to an inspectable element."""
        assert self.context is not None
        for station in self.context.runtime_topology.stations:
            for selection in get_element_selections(station):
                if selection.grid_model_id == grid_model_id:
                    return station, selection
                if selection.kind == "coupler":
                    coupler = next(coupler for coupler in station.couplers if coupler.grid_model_id == selection.grid_model_id)
                    switch_ids = get_coupler_switch_ids(coupler)
                else:
                    connections = station.branch_connections if selection.kind == "branch" else station.injection_connections
                    switch_ids = get_asset_bay_switch_ids(connections[selection.connection_index])
                if grid_model_id in switch_ids:
                    return station, selection
        return None

    def _open_clicked_element_details(self, grid_model_id: str) -> None:
        """Select a clicked SLD element or its physical switch and open the detail view."""
        if self.context is None or not grid_model_id:
            return
        resolved_element = self._find_clicked_element(grid_model_id)
        if resolved_element is None:
            return
        station, selection = resolved_element
        self.voltage_level.value = station.voltage_level_id
        self.bus_group.value = station.bus_group_id
        self.element.value = selection
        self.tabs.selected_index = 2

    def _bind_clickable_sld_callbacks(self, sld_widget: Any) -> None:
        """Connect Powsybl switch and feeder context-clicks to the element detail view."""
        def open_switch_details(widget: Any) -> None:
            self._open_clicked_element_details(_read_clicked_equipment_id(widget.clicked_switch))

        def open_feeder_details(widget: Any) -> None:
            self._open_clicked_element_details(_read_clicked_equipment_id(widget.clicked_feeder))

        sld_widget.on_switch(open_switch_details)
        sld_widget.on_feeder(open_feeder_details)

    def _render_sld(self, container: widgets.VBox, key: str, focus_ids: set[str]) -> None:
        assert self.context is not None
        if self.clickable_sld.value:
            from pypowsybl_jupyter import display_sld, update_sld

            svg = self._get_clickable_sld(focus_ids)
            sld_widget = self.clickable_sld_widgets[key]
            if sld_widget is None:
                sld_widget = display_sld(svg, enable_callbacks=True, on_hover_func=self._hover_info)
                self._bind_clickable_sld_callbacks(sld_widget)
                self.clickable_sld_widgets[key] = sld_widget
            else:
                update_sld(sld_widget, svg, keep_viewbox=True, enable_callbacks=True)
            container.children = (sld_widget,)
            return
        svg_content = get_sld_content(self.context, self.voltage_level.value, focus_ids)
        self.sld_widgets[key] = update_or_create_sld(self.sld_widgets[key], svg_content, self._hover_info)
        container.children = (self.sld_widgets[key],)

    def _render_voltage_level(self) -> None:
        assert self.context is not None
        focus_ids = get_voltage_level_focus_ids(self.context, self.voltage_level.value)
        self._render_sld(self.voltage_level_sld, "voltage_level", focus_ids)
        rows = []
        for station in get_stations_for_voltage_level(self.context, self.voltage_level.value):
            rows.append(
                {
                    "bus group": station.bus_group_id,
                    "bus-branch buses": ", ".join(station.bus_branch_bus_ids),
                    "busbars": len(station.busbars),
                    "branches": len(station.branch_connections),
                    "injections": len(station.injection_connections),
                    "couplers": len(station.couplers),
                    "model log": " | ".join(station.model_log or []),
                }
            )
        with self.summary_output:
            self.summary_output.clear_output(wait=True)
            display(pd.DataFrame(rows))

    def _render_bus_group(self, station: RuntimeBusGroup) -> None:
        self._render_sld(self.bus_group_sld, "bus_group", get_station_focus_ids(station))
        details = pd.DataFrame(
            [
                ("bus group", station.bus_group_id),
                ("station type", station.station_type),
                ("busbars", ", ".join(busbar.grid_model_id for busbar in station.busbars)),
                ("couplers", ", ".join(coupler.grid_model_id for coupler in station.couplers)),
                ("model log", " | ".join(station.model_log or [])),
            ],
            columns=["field", "value"],
        )
        with self.bus_group_output:
            self.bus_group_output.clear_output(wait=True)
            display(details)
            display(pd.DataFrame(station.asset_switching_table, index=[busbar.grid_model_id for busbar in station.busbars]))

    def _render_element(self, station: RuntimeBusGroup, selection: ElementSelection) -> None:
        assert self.context is not None
        focus_ids = {selection.grid_model_id}
        runtime_payload: Any
        if selection.kind == "coupler":
            runtime_payload = next(coupler for coupler in station.couplers if coupler.grid_model_id == selection.grid_model_id)
            focus_ids.update(get_coupler_switch_ids(runtime_payload))
        else:
            connections = station.branch_connections if selection.kind == "branch" else station.injection_connections
            runtime_payload = connections[selection.connection_index]
            focus_ids.update(get_asset_bay_switch_ids(runtime_payload))
        self._render_sld(self.element_sld, "element", focus_ids)
        master_payload = get_master_element_payload(self.context, station.bus_group_id, selection)
        with self.master_output:
            self.master_output.clear_output(wait=True)
            display(HTML("<h4>Master data</h4>"))
            display(as_key_value_frame(master_payload))
        with self.runtime_output:
            self.runtime_output.clear_output(wait=True)
            display(HTML("<h4>Runtime data</h4>"))
            display(as_key_value_frame(runtime_payload))

    def _run_simplification_analysis(self, _button: widgets.Button) -> None:
        """Reproduce simplification for the visible voltage level without mutating loaded data."""
        if self.context is None:
            return
        stations = get_stations_for_voltage_level(self.context, self.voltage_level.value)
        for station in stations:
            self.simplification_cache[station.bus_group_id] = simplify_station_for_analysis(station)
        self._render_simplification_summary()
        self._render_simplification_detail()

    def _render_simplification_summary(self) -> None:
        if self.context is None:
            return
        stations = get_stations_for_voltage_level(self.context, self.voltage_level.value)
        with self.simplification_summary_output:
            self.simplification_summary_output.clear_output(wait=True)
            if not stations:
                display(HTML("<b>No simplification comparison is available because this voltage level has no materialized bus group.</b>"))
                return
            if not self.simplification_cache:
                display(HTML("Run the in-memory analysis to compare the current runtime topology with its simplified view."))
                return
            rows = []
            for station in stations:
                comparison = self.simplification_cache.get(station.bus_group_id)
                if comparison is None:
                    continue
                rows.append(
                    {
                        "bus group": station.bus_group_id,
                        "busbars": f"{len(comparison.original_station.busbars)} -> {len(comparison.simplified_station.busbars)}",
                        "couplers": f"{len(comparison.original_station.couplers)} -> {len(comparison.simplified_station.couplers)}",
                        "branches": f"{len(comparison.original_station.branch_connections)} -> {len(comparison.simplified_station.branch_connections)}",
                        "injections": f"{len(comparison.original_station.injection_connections)} -> {len(comparison.simplified_station.injection_connections)}",
                        "changes": len(comparison.events),
                        "diagnostics": bool(comparison.problems),
                    }
                )
            display(pd.DataFrame(rows))

    def _clear_simplification_view(self) -> None:
        self.simplification_sld.children = ()
        self.simplification_events_output.clear_output()
        self.simplification_original_output.clear_output()
        self.simplification_result_output.clear_output()
        self.simplification_summary_output.clear_output()

    def _render_simplification_detail(self) -> None:
        station = self._selected_simplification_station()
        if station is None:
            return
        comparison = self.simplification_cache.get(station.bus_group_id)
        if comparison is None:
            self.simplification_sld.children = ()
            self.simplification_events_output.clear_output()
            self.simplification_original_output.clear_output()
            self.simplification_result_output.clear_output()
            return
        focus_ids = get_station_focus_ids(comparison.original_station)
        focus_ids.update(get_station_focus_ids(comparison.simplified_station))
        self._render_sld(self.simplification_sld, "simplification", focus_ids)
        event_rows = [
            {
                "category": event.category,
                "object type": event.object_type,
                "grid-model id": event.grid_model_id,
                "retained id": event.retained_grid_model_id or "",
                "reason": event.reason,
            }
            for event in comparison.events
        ]
        with self.simplification_events_output:
            self.simplification_events_output.clear_output(wait=True)
            display(HTML("<h4>Simplification changes</h4>"))
            display(pd.DataFrame(event_rows) if event_rows else pd.DataFrame([{"status": "No structural change detected."}]))
        with self.simplification_original_output:
            self.simplification_original_output.clear_output(wait=True)
            display(HTML("<h4>Original runtime view</h4>"))
            display(as_json_view(comparison.original_station))
        with self.simplification_result_output:
            self.simplification_result_output.clear_output(wait=True)
            display(HTML("<h4>Simplified runtime view</h4>"))
            display(as_json_view(comparison.simplified_station))


def _read_clicked_equipment_id(payload: dict[str, Any]) -> str:
    """Read the grid-model id emitted by a Powsybl-Jupyter switch or feeder callback."""
    return str(payload.get("id") or payload.get("equipment_id") or payload.get("equipmentId") or "")


explorer = AssetTopologyExplorer()
explorer.show()
explorer._load_context(explorer.load_button)

In [ ]:
def get_station_object_name(station: RuntimeBusGroup, object_type: str, grid_model_id: str) -> str:
    """Return an object's display name, falling back to its stable grid-model id."""
    normalized_object_type = "coupler" if object_type == "disconnector" else object_type
    if normalized_object_type == "busbar":
        objects = station.busbars
    elif normalized_object_type == "coupler":
        objects = station.couplers
    elif normalized_object_type == "branch":
        objects = [connection.asset for connection in station.branch_connections]
    elif normalized_object_type == "injection":
        objects = [connection.asset for connection in station.injection_connections]
    elif normalized_object_type == "asset":
        objects = [
            *(connection.asset for connection in station.branch_connections),
            *(connection.asset for connection in station.injection_connections),
        ]
    else:
        return grid_model_id

    matching_object = next((item for item in objects if item.grid_model_id == grid_model_id), None)
    return matching_object.name if matching_object is not None and matching_object.name else grid_model_id


def format_station_object_label(station: RuntimeBusGroup, object_type: str, grid_model_id: str) -> str:
    """Format a station object as a readable name plus its stable identifier."""
    name = get_station_object_name(station, object_type, grid_model_id)
    return grid_model_id if name == grid_model_id else f"{name} [{grid_model_id}]"


def render_bus_group_with_names(self: AssetTopologyExplorer, station: RuntimeBusGroup) -> None:
    """Render bus-group tables with both object names and stable identifiers."""
    self._render_sld(self.bus_group_sld, "bus_group", get_station_focus_ids(station))
    details = pd.DataFrame(
        [
            ("bus group", station.bus_group_id),
            ("station type", station.station_type),
            (
                "busbars",
                ", ".join(
                    format_station_object_label(station, "busbar", busbar.grid_model_id) for busbar in station.busbars
                ),
            ),
            (
                "couplers",
                ", ".join(
                    format_station_object_label(station, "coupler", coupler.grid_model_id) for coupler in station.couplers
                ),
            ),
            ("model log", " | ".join(station.model_log or [])),
        ],
        columns=["field", "value"],
    )
    with self.bus_group_output:
        self.bus_group_output.clear_output(wait=True)
        display(details)
        display(
            pd.DataFrame(
                station.asset_switching_table,
                index=[
                    format_station_object_label(station, "busbar", busbar.grid_model_id)
                    for busbar in station.busbars
                ],
            )
        )


def render_simplification_detail_with_setup_status(self: AssetTopologyExplorer) -> None:
    """Render structural changes and retained objects with their names and identifiers."""
    station = self._selected_simplification_station()
    if station is None:
        return
    comparison = self.simplification_cache.get(station.bus_group_id)
    if comparison is None:
        self.simplification_sld.children = ()
        self.simplification_events_output.clear_output()
        self.simplification_original_output.clear_output()
        self.simplification_result_output.clear_output()
        return

    focus_ids = get_station_focus_ids(comparison.original_station)
    focus_ids.update(get_station_focus_ids(comparison.simplified_station))
    svg_content = get_sld_content(self.context, self.voltage_level.value, focus_ids)
    missing_sld_ids = {
        grid_model_id for grid_model_id in focus_ids if not is_grid_model_id_in_sld(svg_content, grid_model_id)
    }
    self._render_sld(self.simplification_sld, "simplification", focus_ids)

    change_rows = [
        {
            "status": event.category,
            "object type": event.object_type,
            "asset name": get_station_object_name(
                comparison.original_station, event.object_type, event.grid_model_id
            ),
            "grid-model id": event.grid_model_id,
            "retained name": (
                get_station_object_name(comparison.simplified_station, "busbar", event.retained_grid_model_id)
                if event.retained_grid_model_id
                else ""
            ),
            "retained id": event.retained_grid_model_id or "",
            "reason": event.reason,
            "SLD": "not applicable",
        }
        for event in comparison.events
    ]
    simplified_ids = get_station_object_ids(comparison.simplified_station)
    retained_rows = [
        {
            "status": "retained",
            "object type": object_type,
            "asset name": get_station_object_name(comparison.simplified_station, object_type, grid_model_id),
            "grid-model id": grid_model_id,
            "retained name": "",
            "retained id": "",
            "reason": "part of simplified setup",
            "SLD": "not represented by Powsybl" if grid_model_id in missing_sld_ids else "represented",
        }
        for object_type, grid_model_ids in simplified_ids.items()
        for grid_model_id in sorted(grid_model_ids)
    ]
    with self.simplification_events_output:
        self.simplification_events_output.clear_output(wait=True)
        display(HTML("<h4>Simplification changes and retained setup</h4>"))
        display(pd.DataFrame([*change_rows, *retained_rows]))
        if missing_sld_ids:
            display(
                HTML(
                    "<b>SLD limitation:</b> Powsybl does not render these highlighted topology ids: "
                    + ", ".join(sorted(missing_sld_ids))
                )
            )
    with self.simplification_original_output:
        self.simplification_original_output.clear_output(wait=True)
        display(HTML("<h4>Original runtime view</h4>"))
        display(as_json_view(comparison.original_station))
    with self.simplification_result_output:
        self.simplification_result_output.clear_output(wait=True)
        display(HTML("<h4>Simplified runtime view</h4>"))
        display(as_json_view(comparison.simplified_station))


AssetTopologyExplorer._render_bus_group = render_bus_group_with_names
AssetTopologyExplorer._render_simplification_detail = render_simplification_detail_with_setup_status
explorer._render_bus_group(explorer._selected_station())
explorer._render_simplification_detail()

## 7. Verify results and add tests

In [ ]:
def test_context_has_selectable_station(context: TopologyContext) -> None:
    """Check that one validated context exposes at least one station and element selection."""
    assert context.runtime_topology.stations
    station = context.runtime_topology.stations[0]
    assert station.bus_group_id in {master_station.bus_group_id for master_station in context.master_data.stations}
    assert get_element_selections(station)


def test_master_runtime_payloads_stay_separate(context: TopologyContext) -> None:
    """Check that selected runtime elements resolve a canonical payload without type coercion."""
    station = context.runtime_topology.stations[0]
    selection = get_element_selections(station)[0]
    master_payload = get_master_element_payload(context, station.bus_group_id, selection)
    assert master_payload
    assert selection.grid_model_id in json.dumps(master_payload)


def test_sld_focus_content(context: TopologyContext) -> None:
    """Check that native Powsybl SLDs preserve their SVG root and viewport."""
    voltage_level_id = get_voltage_levels(context).index[0]
    station = get_stations_for_voltage_level(context, voltage_level_id)[0]
    station_focus_ids = get_station_focus_ids(station)
    voltage_level_focus_ids = get_voltage_level_focus_ids(context, voltage_level_id)
    assert station_focus_ids
    assert station_focus_ids <= voltage_level_focus_ids
    svg_content = get_sld_content(context, voltage_level_id, station_focus_ids)
    root = ElementTree.fromstring(svg_content)
    assert root.tag.endswith("svg")
    assert root.get("viewBox")


def test_coupler_switch_focus(context: TopologyContext) -> None:
    """Check that a coupler SLD accepts its structured switch metadata as focus ids."""
    station = next(station for station in context.runtime_topology.stations if station.couplers)
    coupler = station.couplers[0]
    switch_ids = get_coupler_switch_ids(coupler)
    assert all(isinstance(switch_id, str) for switch_id in switch_ids)
    svg_content = get_sld_content(context, station.voltage_level_id, {coupler.grid_model_id, *switch_ids})
    assert ElementTree.fromstring(svg_content).tag.endswith("svg")


def test_simplification_comparison(context: TopologyContext) -> None:
    """Check that the simplification summary remains typed and non-mutating."""
    summary = summarize_simplification(context)
    assert len(summary) == 1
    assert summary.loc[0, "bus group"]


def test_explorer_tab_rendering(app: AssetTopologyExplorer) -> None:
    """Check that every tab embeds the selected static or interactive SLD renderer."""
    app._load_context(app.load_button)
    assert app.context is not None
    assert app.voltage_level.options
    assert app.bus_group.options
    assert app.element.options
    assert app.simplification_bus_group.options
    comparison_frame = run_explorer_simplification_analysis(app)
    assert not comparison_frame.empty
    sld_containers = {
        "voltage_level": app.voltage_level_sld,
        "bus_group": app.bus_group_sld,
        "element": app.element_sld,
        "simplification": app.simplification_sld,
    }
    for tab_index, (sld_key, container) in enumerate(sld_containers.items()):
        app.tabs.selected_index = tab_index
        sld_widget = app.clickable_sld_widgets[sld_key] if app.clickable_sld.value else app.sld_widgets[sld_key]
        assert sld_widget is not None
        assert container.children == (sld_widget,)
        if isinstance(sld_widget, widgets.Image):
            assert sld_widget.layout.width == "100%"
            assert sld_widget.layout.height == "560px"
        else:
            assert type(sld_widget).__name__ == "SldWidget"
            assert sld_widget.diagram_data["value_meta"]
    assert app.simplification_cache
    app._load_context(app.load_button)
    assert not app.simplification_cache


def test_clickable_sld_navigation(app: AssetTopologyExplorer) -> None:
    """Check that native SLD callbacks route a coupler to the element detail tab."""
    assert app.context is not None
    station = next(station for station in app.context.runtime_topology.stations if station.couplers)
    coupler_id = station.couplers[0].grid_model_id
    clickable_sld_before_test = app.clickable_sld.value
    app.clickable_sld.value = True
    try:
        app.voltage_level.value = station.voltage_level_id
        app.bus_group.value = station.bus_group_id
        app._render_bus_group(station)
        sld_widget = app.clickable_sld_widgets["bus_group"]
        assert type(sld_widget).__name__ == "SldWidget"
        assert sld_widget.diagram_data["value_meta"]
        app._open_clicked_element_details(coupler_id)
        assert app.element.value.grid_model_id == coupler_id
        assert app.tabs.selected_index == 2
    finally:
        app.clickable_sld.value = clickable_sld_before_test


test_context_has_selectable_station(example_context)
test_master_runtime_payloads_stay_separate(example_context)
test_sld_focus_content(example_context)
test_coupler_switch_focus(example_context)
test_simplification_comparison(example_context)
test_explorer_tab_rendering(explorer)
test_clickable_sld_navigation(explorer)
display(HTML("<b>Asset-topology notebook checks passed.</b>"))

In [ ]:
def test_switch_callback_routes_to_owning_element(app: AssetTopologyExplorer) -> None:
    """Check that a physical SLD switch opens its owning asset or coupler details."""
    assert app.context is not None
    for station in app.context.runtime_topology.stations:
        for selection in get_element_selections(station):
            if selection.kind == "coupler":
                coupler = next(coupler for coupler in station.couplers if coupler.grid_model_id == selection.grid_model_id)
                switch_ids = get_coupler_switch_ids(coupler)
            else:
                connections = station.branch_connections if selection.kind == "branch" else station.injection_connections
                switch_ids = get_asset_bay_switch_ids(connections[selection.connection_index])
            if switch_ids:
                expected_station = station
                expected_selection = selection
                clicked_switch_id = next(iter(switch_ids))
                break
        else:
            continue
        break
    else:
        raise AssertionError("The test topology does not contain a physical bay switch.")

    app.voltage_level.value = expected_station.voltage_level_id
    app.bus_group.value = expected_station.bus_group_id
    app.tabs.selected_index = 1
    sld_widget = app.clickable_sld_widgets["bus_group"]
    assert sld_widget is not None
    sld_widget.clicked_switch = {"id": clicked_switch_id}
    sld_widget.on_switch_msg()

    assert app.element.value == expected_selection
    assert app.tabs.selected_index == 2


test_switch_callback_routes_to_owning_element(explorer)
print("Physical switch callbacks open the owning element details.")

In [ ]:
def test_retained_coupler_is_visible_in_setup_status(context: TopologyContext) -> None:
    """Check that a retained coupler receives the SLD highlight class."""
    for station in context.runtime_topology.stations:
        comparison = simplify_station_for_analysis(station)
        simplified_coupler_ids = get_station_object_ids(comparison.simplified_station)["coupler"]
        svg_content = get_sld_content(context, station.voltage_level_id, get_station_focus_ids(station))
        svg_elements = {element.get("id"): element for element in ElementTree.fromstring(svg_content).iter()}
        for coupler_id in simplified_coupler_ids:
            coupler_svg_id = "id" + "".join(character if character.isalnum() else f"_{ord(character)}_" for character in coupler_id)
            coupler_element = svg_elements.get(coupler_svg_id)
            if coupler_element is not None:
                assert coupler_id not in {event.grid_model_id for event in comparison.events}
                assert is_grid_model_id_in_sld(svg_content, coupler_id)
                assert "highlight" in coupler_element.get("class", "")
                return
    raise AssertionError("The topology does not contain a retained coupler visible in the SLD.")


test_retained_coupler_is_visible_in_setup_status(example_context)
print("A retained coupler is highlighted in the SLD.")

## 8. Usage and limitations

- Select a processed folder to use its canonical master data. If no runtime topology was saved, it is materialized from the loaded Powsybl network state.
- Select an XIIDM file to derive the master and runtime topologies from the Powsybl network exclusively in memory.
- The diagram tabs use `pypowsybl_jupyter`: the active element and its known switches are highlighted, while other voltage-level elements are muted.
- The view does not change switch states in the network. It shows the state present at load time; reload the network for an updated runtime view.
- The **Simplification** tab reruns the current preprocessing simplification with `close_couplers=False` exclusively in memory. It compares the original runtime bus group with the simplified runtime view and identifies only structural differences supported by stable IDs or production diagnostics.
- The simplification view is not a persisted historical audit trail. Results may differ from an earlier preprocessing run if the code, parameters, or initial network state have changed since then.